# Databricks notebook markdown documentation

# 📊 Data Skewness in Apache Spark (Complete Guide for Databricks)

This document explains:
- What is data skewness
- Why it happens
- How to detect it
- How to fix it using:
  - Salting
  - Broadcast joins
  - AQE (Adaptive Query Execution)

---

# 📌 1. What is Data Skewness?

In Spark, **data skewness** happens when data is **unevenly distributed across partitions**, causing some tasks to process much more data than others.

👉 Spark tries to divide data equally, but skew breaks this balance.

---

## 📉 Example of Skewed Distribution


- Partition 1 → 5 GB 🔥 (very heavy)
- Partition 2 → 200 MB
- Partition 3 → 150 MB
- Partition 4 → 100 MB


**❌ Why this is a problem**
- One task becomes a bottleneck
- Entire job waits for slow task
- Executor may crash (OOM)
- High shuffle spill (disk usage increases)
- Poor cluster utilization

**📍 Where skew happens most**
- Joins (key-based joins)
- GroupBy / Aggregations
- Shuffle operations

---

# ⚠️ 2. Symptoms of Data Skew

**🔴 In Spark UI**
- One task runs significantly longer
- Stage stuck at 90–99%
- One executor heavily loaded

**📉 Execution Symptoms**
- Slow job completion
- Uneven shuffle read/write
- Executor memory overflow (OOM)

---

# 🔥 3. Data Skew Handling Techniques

## 🧂 3.1 Salting (Manual Fix)

**🧠 Idea:**  
Break a hot key into multiple smaller keys so data is distributed evenly.

**❌ Problem Example:**  
`user_id = 100` appears 1,000,000 times  
All data goes to one partition → skew occurs.

**🛠️ Solution: Add Salt**  
Modify the key:  
`user_id → user_id + random_number`

**Example:**  
- 100 → 100_0
- 100 → 100_1
- 100 → 100_2

**📊 Before Salting**
- Partition A → 1,000,000 rows (user 100)
- Partition B → small data
- Partition C → small data

**📊 After Salting**
- Partition A → 300,000 rows
- Partition B → 350,000 rows
- Partition C → 350,000 rows

✔ Even distribution  
✔ No single bottleneck

**📌 When to use Salting**
- Extreme skew in joins
- One key dominates dataset

**⚠️ Limitation**
- More complex logic
- Requires additional join handling

---

## 📡 3.2 Broadcast Joins (Best Performance Fix)

**🧠 Idea:**  
Send small table to all executors instead of shuffling large data.

**❌ Problem Scenario:**  
`orders JOIN users ON user_id`  
- orders → billions of rows  
- users → small table

**❌ Normal Join Problem**
- Full shuffle happens
- Skew possible
- Expensive network IO

**✅ Broadcast Join Solution**  
`broadcast(users)`

**📊 What happens**
- Small table copied to all executors
- No shuffle for small table
- Join happens locally

**🚀 Benefits**
- No shuffle
- Faster execution
- Eliminates skew risk

**⚠️ Limitation**
- Only works if one table is small (usually < 100 MB)

---

## ⚡ 3.3 AQE (Adaptive Query Execution) ⭐

**🧠 Idea:**  
Spark automatically detects and fixes skew at runtime.

**🔧 Enable AQE**  
`spark.sql.adaptive.enabled=true`

**⚙️ What AQE does**
1. Detects skewed partitions  
   Identifies uneven data distribution during execution.
2. Splits large partitions  
   - Before: 1 large partition → 1 task  
   - After: 1 large partition → multiple smaller tasks
3. Optimizes joins automatically  
   - Detects skewed join keys  
   - Splits heavy partitions  
   - Rebalances workload

**📊 Before AQE**
- Task 1 → 10 GB 🔥 (slow)
- Task 2 → 200 MB
- Task 3 → 300 MB

**📊 After AQE**
- Task 1 → split into smaller tasks (2 GB each)
- Task 2 → 200 MB
- Task 3 → 300 MB

✔ Balanced execution  
✔ Faster job completion

**🚀 Why AQE is powerful**
- No code changes required
- Runtime optimization
- Handles skew dynamically

**⚠️ Limitation**
- Best in Spark 3+
- Extreme skew may still need salting

---

# 🧠 4. Comparison of Techniques

| Technique       | When to Use         | Effort | Effectiveness |
|-----------------|--------------------|--------|--------------|
| Salting         | Heavy skew keys     | High   | 🔥🔥🔥        |
| Broadcast Join  | Small table joins   | Low    | 🔥🔥🔥🔥      |
| AQE             | General workloads   | None   | 🔥🔥🔥        |

---

# 🎯 5. Summary

👉 Data skewness is an imbalance in data distribution that causes slow Spark jobs.

**Fix strategies:**
- 🧂 Salting → manually split hot keys
- 📡 Broadcast Join → avoid shuffle
- ⚡ AQE → Spark automatically fixes skew

🚀 **Final Insight**  
If one task is much slower than others, your Spark job is almost certainly suffering from data skewness.

<span style="font-size:90%">

---

## 🧂 **Salting Technique to Resolve Data Skewness in Spark**

---

### **1. What is Salting?**
Salting is a method to break up a **hot key** (a key that appears very frequently and causes skew) by adding a random "salt" value to the key, distributing the data more evenly across partitions.

---

### **2. Example Scenario**
Suppose we have a dataset where `user_id=100` appears 1,000,000 times, causing skew in joins.

- **Problem:** All records with `user_id=100` go to one partition, making the join slow.

---

### **3. Solution Steps**

#### **Step 1: Add Salt to the Skewed DataFrame (`orders`)**
For hot key `user_id=100`, add a random salt between 0 and N-1 (e.g., N=3).

python
from pyspark.sql.functions import col, lit, rand, floor

orders = spark.createDataFrame([
    (100, "orderA"), (100, "orderB"), (100, "orderC"), (101, "orderD")
], ["user_id", "order_id"])

N = 3
orders_salted = orders.withColumn(
    "salt", floor(rand(seed=42) * N)
)


---

#### **Step 2: Expand the Users DataFrame for the Hot Key**
For `user_id=100`, create rows for each salt value.

python
from pyspark.sql.functions import explode, array

users = spark.createDataFrame([
    (100, "Alice"), (101, "Bob")
], ["user_id", "user_name"])

users_expanded = users.withColumn(
    "salt",
    explode(
        array([lit(i) for i in range(N)]) if col("user_id") == 100 else array([lit(0)])
    )
)


---

#### **Step 3: Join on Both `user_id` and `salt`**

python
result = orders_salted.join(
    users_expanded,
    on=["user_id", "salt"],
    how="inner"
)
display(result)


---

### **4. Summary**
- **Salting** breaks up skewed keys by adding a random salt.
- The join is performed on both the original key and the salt.
- This distributes the data more evenly and avoids bottlenecks.

</span>